# Energy networks

Optionally build the energy networks, then inspect one of them: summary, map, node and edge tables, validation report and PyPSA network.

The workflow turns the inputs checked in `00_inputs.ipynb` into three networks, each a PyPSA file with GeoParquet map layers:

- `base-mauritius`: CEB substations moved onto the CEB 66 kV routes, with the power plants attached.
- `inferred-osm-<region>`: OpenStreetMap substations and power plants, connected by the roads near lit areas (the estimated distribution network).
- `inferred-provided-<region>`: CEB substations, power plants and 66 kV lines, connected by the same roads.

`<region>` is `region` in `config/energy/energy.yaml` (`mauritius-rodrigues`). Voltages and capacities of the inferred lines are placeholders: `v_nom_kv` is empty in the map layers and `model_v_nom_kv` holds the value the model uses.

In [ ]:
import pathlib
import subprocess
import sys

for _candidate in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_candidate / "_helpers.py").exists():
        sys.path.insert(0, str(_candidate))
        break
# _helpers finds the workflow's files and draws them.
import _helpers as h  # noqa: E402

print("data root: ", h.DATA_ROOT)
print("model data:", h.MODEL_DATA)

## How the workflow runs

This notebook builds nothing unless you set `RUN_BUILD = True` in the next cell. The target `build_energy_networks` in `workflow/0-preprocess/energy.smk` asks for the three `<network>.nc` files, and Snakemake runs these rules:

1. `fetch_energy_osm` downloads the OpenStreetMap roads, power features and island outlines, if a file is missing.
2. `prepare_energy_assets` cleans the CEB data into `data/processed/energy/<model_data>/provided/` and places the plants of the CEB Annual Report.
3. `build_base_energy_network` writes `base-mauritius`.
4. `fetch_energy_nightlights` (if an image is missing), `build_energy_nightlight_composite` and `build_energy_nightlight_targets` turn the monthly VIIRS images into lit-pixel points (`targets.geoparquet`).
5. `build_inferred_energy_network` runs twice, writing `inferred-osm-<region>` and `inferred-provided-<region>`. Both read the roads and targets; the osm network also reads the OpenStreetMap power features, the provided network the cleaned CEB tables.

A network is written to `data/processed/energy/<model_data>/networks/<name>/` (`<name>.nc`, `<name>_metadata.json`, and the `geoparquet/` layers this notebook reads) and to `data/out/energy/<model_data>/<name>/` (`generators.csv`, `lines.csv`, `validation.json`). `<model_data>` is `model_data` in `config/energy/energy.yaml`.

The demand shares of `inferred-provided` are a separate target: the `build_energy_demand` rule writes them to `data/processed/energy/<model_data>/demand/`. "Running the model" in `docs/src/infrastructure-energy.md` gives its command, and the command for one network.

From the terminal (repository root, `mu-star` environment):

```shell
snakemake -n -c1 build_energy_networks  # dry run: what would run, and why
snakemake -c1 build_energy_networks     # the three networks
```

In [ ]:
# Set RUN_BUILD = True to run Snakemake from the repository root with this kernel's Python.
RUN_BUILD = False
# All three networks. For one network, give the path of the file it writes, relative to the repository root.
TARGET = "build_energy_networks"
EXTRA_ARGS: list[str] = []  # for example ["-n"] for a dry run

if RUN_BUILD:
    subprocess.run([sys.executable, "-m", "snakemake", "-c1", TARGET, *EXTRA_ARGS], cwd=h.REPO_ROOT, check=True)

## Pick a network

The next cell lists the built networks (those with `geoparquet/` layers and a manifest) and picks one by name. Change `NETWORK` and rerun the cells below to see another.

In [ ]:
networks = h.available_networks()
if not networks:
    raise RuntimeError(f"No built networks under {h.NETWORKS_DIR}. Run `{h.BUILD_ALL}` or set RUN_BUILD = True above.")
display(h.list_networks())

NETWORK = f"inferred-provided-{h.REGION_SLUG}"  # or another name from the table
if NETWORK not in networks:
    raise ValueError(f"{NETWORK} is not built; pick one of {networks}")
print("Selected NETWORK =", NETWORK)

## Summary

Counts, total line length, and which electrical values are placeholders (from the manifest).

In [ ]:
summary = h.summarise(NETWORK)
for key in ("network", "methodology", "nodes", "edges", "line_length_km", "crs", "electrical_values_note"):
    print(f"{key:24s} {summary[key]}")
print("\nnodes by kind:  ", summary["node_kinds"])
print("edges by source:", summary["edge_sources"])

## Interactive map

Hover over a bus to see its attributes (Plotly on an OpenStreetMap basemap). Every edge is drawn, coloured by `source`; in the inferred networks the roads are the estimated distribution lines, drawn under the 66 kV lines and connections. Only substations and generators are marked. Use `roads=False` to hide the roads, or `clip=h.MAURITIUS_BBOX` to show one island (Rodrigues is about 560 km from Mauritius, so the full view is mostly sea). `h.plot_network(NETWORK)` draws a static matplotlib version.

In [ ]:
h.explore_network(NETWORK)

## Node and edge tables

These GeoParquet layers are what the viewer shows. Nodes have `bus_id`, `kind` (`substation`, `generator`, `transformer`, `distribution_node`, or `junction` in base-mauritius), `source` and `model_v_nom_kv`; a power asset connected to a road also has `anchor_status` and `anchor_distance_m` (its distance from that road). Edges have `source` (for example `osm` for roads, `provided_transmission`, and `substation_anchor` or `generator_anchor` for connections), `bus0`, `bus1`, `length_km` and `model_v_nom_kv`. `is_inferred` marks what the estimate added. In the inferred networks `v_nom_kv` and `s_nom_mva` are empty; the placeholder values are in `model_v_nom_kv` and `model_s_nom_mva`. The first table shows the substations and plants: `h.power_nodes` drops the road points and transformers.

In [ ]:
nodes, edges = h.load_layers(NETWORK)
node_columns = [
    "bus_id",
    "name",
    "kind",
    "source",
    "model_v_nom_kv",
    "anchor_status",
    "anchor_distance_m",
    "is_inferred",
]
edge_columns = ["line_id", "source", "bus0", "bus1", "length_km", "model_v_nom_kv", "model_s_nom_mva", "is_inferred"]
print("Substations and plants:")
display(h.power_nodes(nodes)[node_columns].head(10))
print("Edges:")
display(edges[edge_columns].head(10))
print("Edges by source:")
display(edges.groupby("source", dropna=False)["length_km"].agg(count="size", length_km="sum").round(1))

## Validation report

`data/out/energy/<model_data>/<network>/validation.json` has totals, an `errors` list (empty for a usable network) and `warnings`, for example when the line length or generation capacity differs from CEB's published totals by more than the set tolerance. The same warnings are logged when the rules run. `status` is `valid`, `valid_with_warnings` or `invalid`.

In [ ]:
if h.found(h.OUT_DIR / NETWORK / "validation.json", h.BUILD_ALL):
    validation = h.load_validation(NETWORK)
    print("status:", validation["status"])
    print("totals:", validation["totals"])
    for error in validation.get("errors", []):
        print("  error:  ", error)
    for warning in validation.get("warnings", []):
        print("  warning:", warning)
    for check, result in validation.get("checks", {}).items():
        print(f"  check {check}: {result.get('status')}")

## PyPSA view

The same network as a PyPSA file (`<network>.nc`): buses, lines, generators, and transformers where inferred-provided joins the CEB 66 kV lines to the 11 kV roads. Capacities are fixed; nothing is extendable.

In [ ]:
n = h.load_pypsa(NETWORK)
print(n)
print(f"{len(n.buses)} buses, {len(n.lines)} lines, {len(n.transformers)} transformers, {len(n.generators)} generators")
display(n.buses.head())
display(n.lines.head())